# Veritas PoC on Google Colab

1. `Runtime > Change runtime type > T4 GPU`
2. Run cells **in order**, one at a time.
3. **Do not click Run on another cell while one is still executing.** Colab automatically sends `^C` to the running process — you do not have to press anything.
4. Cell 6 takes ~15–30 min total. You should see progress like `generating...`, `caching activations...`, `saved checkpoint`.

HF: accept https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct then paste token in cell 4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
if os.path.basename(os.getcwd()) != 'algoverse-PoC':
    if not os.path.isdir('algoverse-PoC'):
        !git clone https://github.com/abdelmagid07/algoverse-PoC.git
    %cd algoverse-PoC
!git pull --ff-only

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token: '))

In [ ]:
# Optional: see what's already done (safe to re-run after an interrupt)
from pathlib import Path
for name, p in [
    ('trajectories', 'results/trajectories.json'),
    ('fast scores', 'results/fast_scores.json'),
    ('slow scores', 'results/slow_scores.json'),
    ('correlation', 'results/correlation.json'),
    ('summary', 'results/poc_summary.md'),
]:
    path = Path(p)
    extra = ''
    if path.exists() and p.endswith('.json'):
        import json
        data = json.loads(path.read_text())
        extra = f' ({len(data)} entries)' if isinstance(data, (list, dict)) else ''
    print(f"{'OK' if path.exists() else 'MISSING':7} {name}{extra}")

In [ ]:
# ONE cell for the whole pipeline — model loads once, checkpoints after each trajectory.
# If interrupted, re-run THIS cell only; it resumes from partial trajectories.json.
!python run_pipeline.py

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

for p in ['results/correlation.json', 'results/importance_by_step.png', 'results/poc_summary.md']:
    assert Path(p).exists(), f'Missing {p} — run cell 6 and wait for "=== Done ==="'

print(json.dumps(json.load(open('results/correlation.json')), indent=2))
display(Image('results/importance_by_step.png'))
display(Markdown(open('results/poc_summary.md').read()))